In [0]:
%run /Workspace/Users/rahulpatel@cyntexa.com/IAP_Practice/voltpath/00_setup

In [0]:
catalog = dbutils.widgets.get("catalog")



In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import col

df = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .load("/Volumes/vp_rp20040126/landing/raw/stations/")
)

stations_df = df.select(*[col(c).cast("string").alias(c) for c in df.columns])

# Add metadata columns (ingestion timestamp, source file, etc.)
stations_df = add_metadata(stations_df)

# Write the bronze table to the parameterized catalog
full_table_name = f"{catalog}.bronze.stations"

(stations_df.write
    .format("delta")
    .mode("overwrite")
    .option("mergeSchema", "true")
    .saveAsTable(full_table_name)
)

In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import col

# Read CSV files from the landing volume (infer schema, then cast all columns to string)
tariffs_df = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .load("/Volumes/vp_rp20040126/landing/raw/tariffs/")
)

# Cast every column to string type
tariffs_df = tariffs_df.select(*[col(c).cast("string").alias(c) for c in tariffs_df.columns])
tariffs_df = add_metadata(tariffs_df)

# Write the bronze table to the parameterized catalog
tariffs_table_name = f"{catalog}.bronze.tariffs"

(tariffs_df.write
    .format("delta")
    .mode("overwrite")
    .option("mergeSchema", "true")
    .saveAsTable(tariffs_table_name)
)

In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import col, lit

member_table = f"{catalog}.bronze.member"
member_path = "/Volumes/vp_rp20040126/landing/raw/members/"

# Read CSV files from the landing volume (infer schema, then cast all columns to string)
member_df = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .load(member_path)
)

# Cast every column to string type
member_df = member_df.select(*[col(c).cast("string").alias(c) for c in member_df.columns])

# Add change_effective_date column (null for snapshot rows; populated in change/CDC files)
member_df = member_df.withColumn("change_effective_date", lit(None).cast("string"))

# Add metadata columns (ingestion timestamp, source file, etc.)
member_df = add_metadata(member_df)

# Write the bronze table to the parameterized catalog
(member_df.write
    .format("delta")
    .mode("overwrite")
    .option("mergeSchema", "true")
    .saveAsTable(member_table)
)

In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import col, lit, struct

chargers_path = "/Volumes/vp_rp20040126/landing/raw/chargers/"
chargers_table = f"{catalog}.bronze.chargers"

# Define explicit schema — all top-level columns are string, connector stays a struct
chargers_schema = StructType([
    StructField("charger_id", StringType(), True),
    StructField("station_id", StringType(), True),
    StructField("connector", StructType([
        StructField("type", StringType(), True),
        StructField("max_kw", StringType(), True)
    ]), True),
    StructField("is_active", StringType(), True),
    StructField("commissioned_on", StringType(), True),
    StructField("snapshot_date", StringType(), True)
])

# Read JSON files — one JSON array per file (multiline=true)
chargers_df = (
    spark.read
    .format("json")
    .option("multiline", "true")
    .schema(chargers_schema)
    .load(chargers_path)
)

# Add metadata columns (ingestion timestamp, source file, etc.)
chargers_df = add_metadata(chargers_df)

# Write the bronze table
(chargers_df.write
    .format("delta")
    .mode("overwrite")
    .option("mergeSchema", "true")
    .saveAsTable(chargers_table)
)

In [0]:
from pyspark.sql.functions import *

sessions_path = "/Volumes/vp_rp20040126/landing/raw/sessions/"
sessions_table = f"{catalog}.bronze.sessions"
checkpoint_path = "/Volumes/vp_rp20040126/ops/pipeline_state/session_checkpoints/"
schema_location = "/Volumes/vp_rp20040126/ops/pipeline_state/session_schemas/"

# Auto Loader: process new JSON files, then stop (availableNow trigger)
# - schemaEvolutionMode=addNewColumns: new fields in later files become new columns automatically
# - rescuedDataColumn: values that don't fit the schema are kept, not lost
# - schemaHints provides the base schema while still allowing schema evolution
sessions_df = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "JSON")
    .option("cloudFiles.schemaLocation", schema_location)
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
    .option("cloudFiles.schemaHints", "session_id STRING, charger_id STRING, member_id STRING, started_at STRING, ended_at STRING, updated_at STRING, status STRING, meter STRUCT<start_kwh: STRING, end_kwh: STRING>, idle_minutes STRING, payment STRUCT<method: STRING, amount_inr: STRING>, faults ARRAY<STRUCT<code: STRING, at: STRING>>")
    .option("rescuedDataColumn", "_rescued_data")
    .load(sessions_path)
)

# Add metadata columns (ingestion timestamp, source file, etc.)
sessions_df = add_metadata(sessions_df)

# Write to bronze table — process new files, then stop
# Auto Loader tracks processed files in checkpoint; re-running with no new files changes nothing
stream = (sessions_df.writeStream
    .format("delta")
    .option("checkpointLocation", checkpoint_path)
    .option("mergeSchema", "true")
    .trigger(availableNow=True)
    .toTable(sessions_table)
)

# Wait for the batch to complete
stream.awaitTermination()